# Pandas Fundamentals

**AI Engineering School — Data Analysis with Python**

Pandas is a Python library for working with structured data such as CSV files, spreadsheets, database tables, and annotation datasets.

In this notebook, you will learn how to:

- create and understand `Series` and `DataFrame` objects;
- inspect a dataset before analyzing it;
- select rows and columns with `[]`, `.loc[]`, and `.iloc[]`;
- filter, sort, and summarize data;
- count categories and create simple cross-tabulations.

All examples are self-contained, so the notebook runs without downloading any external dataset.

## 1. Import Pandas

The standard convention is to import Pandas with the alias `pd`.

In [ ]:
import pandas as pd

print("Pandas version:", pd.__version__)

## 2. The two main Pandas objects

### `Series`

A `Series` is a one-dimensional labeled sequence. You can think of it as one spreadsheet column with an index.

In [ ]:
word_counts = pd.Series(
    [12, 8, 5, 3],
    index=["زبان", "داده", "مدل", "پیکره"],
    name="frequency"
)

word_counts

The labels on the left are the **index**, and the values on the right are the data.

In [ ]:
print("Values:", word_counts.to_numpy())
print("Index:", word_counts.index.tolist())
print("Data type:", word_counts.dtype)

You can select a value by its label. Using `.loc[]` makes label-based selection explicit.

In [ ]:
word_counts.loc["زبان"]

A `Series` can also be created from a dictionary. Dictionary keys become index labels.

In [ ]:
scores = {
    "Ali": 90,
    "Sara": 99,
    "Hassan": 87
}

student_scores = pd.Series(scores, name="score")
student_scores

In [ ]:
student_scores[student_scores >= 90]

### `DataFrame`

A `DataFrame` is a two-dimensional table with labeled rows and columns. It is the most commonly used Pandas object.

In [ ]:
coffee = pd.DataFrame({
    "Day": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
    "Coffee Type": ["Latte", "Espresso", "Latte", "Americano", "Espresso", "Latte", "Americano"],
    "Units Sold": [18, 25, 20, 15, 30, 35, 22],
    "Unit Price": [4.5, 3.0, 4.5, 3.5, 3.0, 4.5, 3.5]
})

coffee

## 3. Understanding the structure of a DataFrame

Before analyzing data, inspect its size, column names, data types, and a few sample rows.

In [ ]:
print("Shape (rows, columns):", coffee.shape)
print("Columns:", coffee.columns.tolist())
print("Index:", coffee.index.tolist())

In [ ]:
coffee.head()

In [ ]:
coffee.tail(2)

In [ ]:
coffee.sample(3, random_state=42)

`random_state` makes random sampling reproducible. Anyone rerunning the notebook will receive the same sample.

In [ ]:
coffee.info()

`info()` is especially useful because it shows:

- the number of non-missing values;
- each column's data type;
- approximate memory usage.

In [ ]:
coffee.describe()

`describe()` summarizes numeric columns. Use `include="all"` when you also want descriptive information for text or categorical columns.

In [ ]:
coffee.describe(include="all")

## 4. Selecting columns

Use single brackets to obtain a `Series` and double brackets to obtain a `DataFrame`.

In [ ]:
coffee["Units Sold"]

In [ ]:
coffee[["Day", "Coffee Type", "Units Sold"]]

Attribute access such as `coffee.Day` sometimes works, but bracket notation is safer because it also supports spaces and unusual column names.

## 5. Selecting rows with `.loc[]` and `.iloc[]`

- `.loc[]` selects by **labels**.
- `.iloc[]` selects by **integer positions**.

In [ ]:
# Row with index label 0
coffee.loc[0]

In [ ]:
# Rows with labels 1, 3, and 5
coffee.loc[[1, 3, 5]]

In [ ]:
# Rows 2 through 4 and selected columns
# Notice that .loc includes both endpoints.
coffee.loc[2:4, ["Day", "Units Sold"]]

In [ ]:
# First three rows and the first two columns by position
# Notice that .iloc follows normal Python slicing and excludes the endpoint.
coffee.iloc[0:3, 0:2]

### Changing the index

A meaningful column can become the index. Avoid directly overwriting `.index` with a column unless you understand the consequences; `set_index()` is clearer.

In [ ]:
coffee_by_day = coffee.set_index("Day")
coffee_by_day

In [ ]:
coffee_by_day.loc["Monday":"Thursday"]

In [ ]:
coffee_by_day.reset_index()

## 6. Filtering rows

A Boolean condition produces `True` or `False` for every row. Pandas keeps the rows where the condition is `True`.

In [ ]:
coffee["Units Sold"] >= 25

In [ ]:
coffee.loc[coffee["Units Sold"] >= 25]

Use `&` for **and**, `|` for **or**, and `~` for **not**. Put each condition inside parentheses.

In [ ]:
coffee.loc[
    (coffee["Units Sold"] >= 20) &
    (coffee["Coffee Type"] == "Latte")
]

In [ ]:
coffee.loc[
    (coffee["Coffee Type"] == "Latte") |
    (coffee["Coffee Type"] == "Espresso")
]

`.isin()` is cleaner when checking several possible values.

In [ ]:
coffee.loc[coffee["Coffee Type"].isin(["Latte", "Espresso"])]

In [ ]:
coffee.loc[coffee["Units Sold"].between(20, 30)]

Text columns also support vectorized string operations through `.str`.

In [ ]:
coffee.loc[coffee["Coffee Type"].str.contains("press", case=False, na=False)]

## 7. Sorting data

`sort_values()` sorts by one or more columns. By default, Pandas returns a new DataFrame and leaves the original unchanged.

In [ ]:
coffee.sort_values("Units Sold", ascending=False)

In [ ]:
coffee.sort_values(
    ["Coffee Type", "Units Sold"],
    ascending=[True, False]
)

## 8. Counting and summarizing categories

In [ ]:
coffee["Coffee Type"].value_counts()

In [ ]:
coffee["Coffee Type"].value_counts(normalize=True).mul(100).round(1)

In [ ]:
print("Unique values:", coffee["Coffee Type"].unique())
print("Number of unique values:", coffee["Coffee Type"].nunique())

A cross-tabulation shows the frequency of combinations of two categorical variables.

In [ ]:
coffee_with_shift = coffee.assign(
    Shift=["Morning", "Morning", "Afternoon", "Afternoon", "Morning", "Afternoon", "Morning"]
)

pd.crosstab(
    coffee_with_shift["Coffee Type"],
    coffee_with_shift["Shift"],
    margins=True
)

## 9. Creating a simple calculated column

Pandas performs arithmetic on entire columns. This is called **vectorized computation**.

In [ ]:
coffee_with_revenue = coffee.assign(
    Revenue=coffee["Units Sold"] * coffee["Unit Price"]
)

coffee_with_revenue

In [ ]:
coffee_with_revenue["Revenue"].sum()

## 10. Reading a CSV file

When your own CSV file is available, use:

```python
df = pd.read_csv("data/my_file.csv")
```

Useful options include:

```python
df = pd.read_csv(
    "data/my_file.csv",
    encoding="utf-8",
    usecols=["text", "label"],
    na_values=["", "NA", "missing"]
)
```

Always check `head()`, `shape`, `info()`, and column names immediately after loading a file.

## 11. Mini practice: a small corpus table

The following table resembles a simple corpus metadata file.

In [ ]:
corpus = pd.DataFrame({
    "text_id": ["T01", "T02", "T03", "T04", "T05", "T06"],
    "genre": ["news", "fiction", "news", "academic", "fiction", "academic"],
    "language": ["Persian", "Persian", "English", "Persian", "English", "Persian"],
    "token_count": [420, 850, 510, 1200, 730, 980],
    "year": [2024, 2023, 2024, 2022, 2024, 2023]
})

corpus

### Practice questions

1. Select only `text_id`, `genre`, and `token_count`.
2. Find Persian texts with at least 900 tokens.
3. Sort the table from the largest to the smallest text.
4. Count the number of texts in each genre.

In [ ]:
# 1
corpus[["text_id", "genre", "token_count"]]

In [ ]:
# 2
corpus.loc[
    (corpus["language"] == "Persian") &
    (corpus["token_count"] >= 900)
]

In [ ]:
# 3
corpus.sort_values("token_count", ascending=False)

In [ ]:
# 4
corpus["genre"].value_counts()

## Key takeaways

- A `Series` is one labeled dimension; a `DataFrame` is a labeled table.
- Inspect a dataset before transforming it.
- Use `.loc[]` for labels and `.iloc[]` for positions.
- Use parentheses around Boolean conditions.
- Prefer vectorized column operations to row-by-row loops.
- Most Pandas operations return a new object unless you assign the result.